Kerchunk Demo
-------------

This is a demonstration on how to use Kerchunk to access data from NOAA Open Data Dissemination (NODD) without downloading the entire dataset. Kerchunk allows you to create a reference file that points to the remote data, enabling you to access only the parts of the data you need.  In this demo, we will use Kerchunk to access a sample dataset from NODD and read it using xarray.

Initial configuration and imports

> This notebook reads GFS data lazily from public AWS S3 by building kerchunk references and opening them with xarray/zarr.

In [4]:
# Configuration for accessing NOAA Open Data Dissemination (NODD) datasets using Kerchunk

# User-selected model: "gfs" or "rtofs"
MODEL_NAME = "gfs"

# Forecast information for the selected model dataset
FORECAST_DATE = "20260501T00Z"
FORECAST_COMPONENT = "atmos"

# Per-model config kwargs used by the model factory.
MODEL_CONFIG_KWARGS = {
    "gfs": {"forecast_component": FORECAST_COMPONENT},
    "rtofs": {},
}

print("Selected model:", MODEL_NAME)
print("Forecast date:", FORECAST_DATE)
print("Model kwargs:", MODEL_CONFIG_KWARGS.get(MODEL_NAME, {}))

Selected model: gfs
Forecast date: 20260501T00Z
Model kwargs: {'forecast_component': 'atmos'}


In [5]:
# Discover available forecast files on S3 (no local download)
from swiftnomads import available_models, get_model_adapter, infer_model_config

if MODEL_NAME not in available_models():
    raise ValueError(f"Unsupported MODEL_NAME '{MODEL_NAME}', choose from {available_models()}")

adapter = get_model_adapter(MODEL_NAME)
cfg = infer_model_config(
    MODEL_NAME,
    FORECAST_DATE,
    **MODEL_CONFIG_KWARGS.get(MODEL_NAME, {}),
)

hours = adapter.list_forecast_hours(cfg, anon=True)
print(f"Discovered {MODEL_NAME.upper()} forecast hours:", hours[:10], "..." if len(hours) > 10 else "")
if not hours:
    raise RuntimeError(f"No forecast files found for model={MODEL_NAME} date={FORECAST_DATE}")

selected_hours = adapter.select_forecast_hours(hours, count=2)
print("Selected forecast hours:", selected_hours)

Discovered GFS forecast hours: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] ...
Selected forecast hours: [0, 1]


In [6]:
# Build/load kerchunk references from a local JSON cache and open lazily
from swiftnomads import get_model_adapter, list_reference_groups, load_reference_metadata

adapter = get_model_adapter(MODEL_NAME)
cache_path = adapter.default_cache_path(cfg, selected_hours)
cached_meta = load_reference_metadata(cache_path)
needs_refresh = adapter.should_refresh_reference(
    cached_meta,
    cfg,
    selected_hours,
    ttl_hours=24,
)

reference = None
reference_error = None
try:
    reference = adapter.get_or_build_refs(
        cfg,
        selected_hours,
        anon=True,
        cache_dir=cache_path.parent,
        ttl_hours=24,
        force_refresh=False,
    )
except ImportError as exc:
    reference_error = exc
    print(
        "Skipping reference build for this model in the current environment:",
        exc,
    )

print("Reference cache:", cache_path)
print("Refresh needed:", needs_refresh)
if reference is not None:
    print("Reference keys:", list(reference.keys()))

new_meta = load_reference_metadata(cache_path)
print(
    "Metadata summary:",
    {
        "model": new_meta.get("model"),
        "built_at": new_meta.get("built_at"),
        "kerchunk_version": new_meta.get("kerchunk_version"),
        "grib2io_available": new_meta.get("grib2io_available"),
        "grib2io_version": new_meta.get("grib2io_version"),
    },
)

if reference is not None:
    groups = list_reference_groups(reference)
    if not groups:
        raise RuntimeError("No zarr groups found in generated kerchunk reference")
    preferred_group = "prmsl/instant/meanSea"
    selected_group = preferred_group if preferred_group in groups else groups[0]
    print("Selected group:", selected_group)

    ds = adapter.open_dataset_cached(
        cfg=cfg,
        forecast_hours=selected_hours,
        cache_dir=cache_path.parent,
        anon=True,
        group=selected_group,
        ttl_hours=24,
        force_refresh=False,
    )
    print(ds)

    # Trigger a tiny read to prove on-demand access.
    if ds.data_vars:
        first_var = next(iter(ds.data_vars))
        tiny = ds[first_var].isel({dim: 0 for dim in ds[first_var].dims}).load()
        print(first_var, tiny.values)
    else:
        print("No data variables were found in this group")
else:
    print(
        "The notebook is configured correctly, but this environment does not have the GFS GRIB2 build dependencies needed to generate kerchunk references."
    )

Dropping unknown variable in msg# 586. Compare with the grib idx file to help identify it and build an ecCodes local grib definitions file to fix it.
Dropping unknown variable in msg# 665. Compare with the grib idx file to help identify it and build an ecCodes local grib definitions file to fix it.
Dropping unknown variable in msg# 1274. Compare with the grib idx file to help identify it and build an ecCodes local grib definitions file to fix it.
Dropping unknown variable in msg# 1285. Compare with the grib idx file to help identify it and build an ecCodes local grib definitions file to fix it.
Dropping unknown variable in msg# 1407. Compare with the grib idx file to help identify it and build an ecCodes local grib definitions file to fix it.
/export/emc-lw-rmahajan/rmahajan/work/src/dau_work/swift-nomads/.conda/lib/python3.14/site-packages/kerchunk/combine.py:403: UserWarning: Concatenated coordinate 'time' contains less than expectednumber of values across the datasets: [1777593600]


Reference cache: .cache/swiftnomads/refs/gfs_bucket-noaa-gfs-bdp-pds_cycle-00_forecast_component-atmos_forecast_date-20260501T00Z_000-001.kerchunk.json
Refresh needed: True
Reference keys: ['refs', 'version']
Metadata summary: {'model': 'gfs', 'built_at': '2026-05-13T13:35:07.225541+00:00', 'kerchunk_version': '0.2.10', 'grib2io_available': True, 'grib2io_version': '2.7.0'}
Selected group: prmsl/instant/meanSea
<xarray.Dataset> Size: 17MB
Dimensions:     (time: 1, step: 2, latitude: 721, longitude: 1440)
Coordinates:
  * time        (time) datetime64[ns] 8B 2026-05-01
  * step        (step) timedelta64[ns] 16B 00:00:00 01:00:00
    valid_time  (time, step) datetime64[ns] 16B ...
  * latitude    (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude   (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
    meanSea     float64 8B ...
Data variables:
    prmsl       (time, step, latitude, longitude) float64 17MB ...
Attributes:
    typeOfLevel:  meanSea
p